# Pratica – Lezione 6: Sequenze e reti ricorrenti

Questo notebook è la pratica che completi autonomamente dopo la Lezione 6 del corso *Advanced Deep Learning*. Qui applichi quanto visto sulla rete neurale ricorrente (RNN) e, in particolare, sulla memoria a lungo e breve termine (LSTM, dall'inglese *Long Short-Term Memory*).

## Cosa farai in questa pratica

Costruirai e userai un generatore di testo carattere per carattere con una LSTM in PyTorch:

- **Esercizio 1**: completi la definizione di una rete LSTM che predice il carattere successivo di un testo.
- **Esercizio 2**: usi quel modello addestrato per generare testo nuovo, carattere per carattere, a partire da un seme.

## Prerequisito

Aver seguito la Lezione 6, "Sequenze e reti ricorrenti", in cui si spiegano lo stato nascosto, la LSTM e le sue porte. Se alcune di quelle equazioni non ti sono familiari, ripassa il capitolo del manuale prima di proseguire.

## Come lavorare con questo notebook

Ogni esercizio ha lo stesso formato: prima il testo, poi una o più celle di codice con parti contrassegnate come `# TODO`, che devi completare tu. Esegui le celle in ordine, dall'alto verso il basso: l'esercizio 2 dipende dal modello che addestri nell'esercizio 1.

Prova a risolvere ogni `TODO` senza guardare la soluzione. Sotto ogni esercizio trovi la soluzione collassata, nel caso ti blocchi o voglia confrontare il tuo approccio.

## Esercizio 1: definisci la LSTM del generatore di testo

**Caso**: vuoi costruire un modello capace di prevedere il carattere successivo di un testo, dato il frammento scritto fino a quel punto. È la stessa idea della tastiera del tuo telefono quando suggerisce la lettera successiva, ma applicata carattere per carattere invece che parola per parola.

Il corpus di addestramento è un testo breve, definito nella stessa cella di codice, sul lavoro di un hub di innovazione digitale. Viene ripetuto più volte perché il modello abbia abbastanza esempi dello stesso schema e possa convergere in poche epoche, cosa comune quando si addestra con un corpus minuscolo.

Hai già scritto tutta l'impalcatura di tokenizzazione, vocabolario e ciclo di addestramento. Il tuo compito è completare la classe `ModeloCharLSTM`: lo strato che converte ogni carattere in un vettore, lo strato ricorrente e lo strato che traduce lo stato nascosto in una previsione sul vocabolario.

**Suggerimenti**:

- Un carattere arriva al modello come indice intero. Per convertirlo in un vettore denso, usa `nn.Embedding(tam_vocab, tam_embedding)`.
- Lo strato ricorrente è `nn.LSTM(tam_embedding, tam_oculto, num_layers=num_capas, batch_first=True)`. L'argomento `batch_first=True` fa sì che la forma dei tensori sia (lotto, tempo, caratteristiche), invece di (tempo, lotto, caratteristiche).
- `nn.LSTM` restituisce due elementi: l'uscita a ogni passo temporale e lo stato finale `(h, c)`. Nel `forward`, propaga entrambi: l'uscita verso lo strato lineare, e lo stato verso chi ha chiamato il modello (ti servirà nell'esercizio 2, per generare testo passo per passo senza rielaborare tutto ciò che è venuto prima).
- Lo strato finale è un `nn.Linear(tam_oculto, tam_vocab)`: converte lo stato nascosto di ogni passo in un vettore di logit, uno per ogni carattere possibile del vocabolario.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# Texto de entrenamiento: se repite para que la LSTM tenga suficientes
# ejemplos del mismo patron y pueda converger en pocas epocas
texto_entrenamiento = (
    "el hub europeo de innovacion digital ayuda a empresas del sector "
    "agroalimentario, forestal y de la salud a adoptar inteligencia artificial. "
    "las empresas que participan en el hub mejoran sus procesos con datos "
    "y con modelos de aprendizaje automatico. "
) * 30

# Vocabulario: cada caracter distinto del texto se convierte en un indice entero
caracteres = sorted(set(texto_entrenamiento))
char_a_idx = {c: i for i, c in enumerate(caracteres)}
idx_a_char = {i: c for i, c in enumerate(caracteres)}
tam_vocab = len(caracteres)

print(f"Tamano del vocabulario: {tam_vocab} caracteres distintos")

# Codificamos todo el texto como un tensor de indices enteros
datos = torch.tensor([char_a_idx[c] for c in texto_entrenamiento], dtype=torch.long)

longitud_secuencia = 40


def crear_lote(datos, longitud_secuencia, tam_lote):
    # Elegimos puntos de inicio aleatorios y recortamos ventanas de longitud fija;
    # el objetivo de cada ventana es la misma ventana desplazada un caracter
    indices_inicio = torch.randint(0, len(datos) - longitud_secuencia - 1, (tam_lote,))
    entradas = torch.stack([datos[i:i + longitud_secuencia] for i in indices_inicio])
    objetivos = torch.stack([datos[i + 1:i + longitud_secuencia + 1] for i in indices_inicio])
    return entradas, objetivos

In [ ]:
class ModeloCharLSTM(nn.Module):
    def __init__(self, tam_vocab, tam_embedding=16, tam_oculto=64, num_capas=1):
        super().__init__()

        # TODO: crea una capa nn.Embedding que convierta cada indice de caracter
        # (entero entre 0 y tam_vocab - 1) en un vector denso de tamano tam_embedding
        self.embedding = ...

        # TODO: crea una capa nn.LSTM que reciba vectores de tamano tam_embedding,
        # produzca un estado oculto de tamano tam_oculto, use num_capas capas
        # apiladas y trabaje con tensores en formato (lote, tiempo, caracteristicas)
        self.lstm = ...

        # TODO: crea una capa nn.Linear que transforme el estado oculto de la LSTM
        # (tamano tam_oculto) en un vector de logits sobre el vocabulario (tamano tam_vocab)
        self.salida = ...

    def forward(self, x, estado=None):
        # TODO: pasa x por la capa de embedding
        x = ...

        # TODO: pasa x (y el estado, si se recibio uno) por la LSTM.
        # nn.LSTM devuelve una tupla (salida_en_cada_paso, estado_final);
        # si estado es None, PyTorch lo inicializa en ceros automaticamente
        salida_lstm, estado = ...

        # TODO: pasa la salida de la LSTM por la capa lineal de salida
        # para obtener los logits sobre el vocabulario en cada paso temporal
        logits = ...

        return logits, estado

In [ ]:
modelo = ModeloCharLSTM(tam_vocab)
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.003)

num_epocas = 300
tam_lote = 32

for epoca in range(num_epocas):
    entradas, objetivos = crear_lote(datos, longitud_secuencia, tam_lote)
    logits, _ = modelo(entradas)
    # reorganizamos (lote, tiempo, vocab) en (lote*tiempo, vocab) para tratar
    # cada posicion de cada secuencia como un ejemplo de clasificacion independiente
    perdida = F.cross_entropy(logits.reshape(-1, tam_vocab), objetivos.reshape(-1))
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()
    if (epoca + 1) % 50 == 0:
        print(f"Epoca {epoca + 1}/{num_epocas} - perdida: {perdida.item():.4f}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
class ModeloCharLSTM(nn.Module):
    def __init__(self, tam_vocab, tam_embedding=16, tam_oculto=64, num_capas=1):
        super().__init__()

        # Converte ogni indice di carattere in un vettore denso di dimensione tam_embedding
        self.embedding = nn.Embedding(tam_vocab, tam_embedding)

        # Strato ricorrente: riceve vettori di dimensione tam_embedding e produce
        # uno stato nascosto di dimensione tam_oculto a ogni passo temporale
        self.lstm = nn.LSTM(tam_embedding, tam_oculto, num_layers=num_capas, batch_first=True)

        # Traduce lo stato nascosto di ogni passo in logit sul vocabolario
        self.salida = nn.Linear(tam_oculto, tam_vocab)

    def forward(self, x, estado=None):
        x = self.embedding(x)
        # estado è la tupla (h, c); se si passa None, PyTorch lo inizializza a zero
        salida_lstm, estado = self.lstm(x, estado)
        logits = self.salida(salida_lstm)
        return logits, estado
```
</details>

## Esercizio 2: genera testo nuovo con il modello addestrato

**Caso**: hai già un modello addestrato che, a partire da un frammento di testo, prevede la distribuzione di probabilità del carattere successivo. Ora vuoi usarlo per generare testo nuovo, carattere per carattere, a partire da un seme che scegli tu (per esempio, `"el hub "`).

L'idea è semplice: dai al modello il seme, ottieni lo stato della LSTM dopo averlo elaborato e, da lì, ripeti per `n_caracteres` volte lo stesso ciclo: prevedi il carattere successivo, lo aggiungi al testo generato e lo usi come input del passo successivo.

Completa la funzione `generar_texto(modelo, texto_inicial, n_caracteres)`. L'impalcatura prepara già lo stato iniziale a partire dal seme; il tuo compito è completare il ciclo che prevede e aggiunge un carattere a ogni iterazione.

**Suggerimenti**:

- Dentro il ciclo, chiama il modello con `ultimo_caracter` (l'ultimo carattere generato, non tutto il testo) e con `estado`. Poiché il modello restituisce lo stato aggiornato, non serve rielaborare il testo completo a ogni passo: è per questo che passare lo stato è tanto utile nella generazione.
- I logit che ti interessano sono quelli dell'ultimo passo temporale: `logits[:, -1, :]`. Trasformali in probabilità con `F.softmax(..., dim=-1)`, dividendo prima per `temperatura` per controllare quanto la distribuzione si affila o si ammorbidisce.
- Per scegliere il carattere successivo non usare sempre il più probabile: usa `torch.multinomial(probabilidades, num_samples=1)`, che campiona secondo quelle probabilità e dà varietà al testo generato.
- Traduci l'indice campionato in carattere con il dizionario `idx_a_char`, che hai già costruito nell'esercizio 1.

In [ ]:
@torch.no_grad()
def generar_texto(modelo, texto_inicial, n_caracteres, temperatura=0.8):
    modelo.eval()

    entrada = torch.tensor([[char_a_idx[c] for c in texto_inicial]], dtype=torch.long)
    _, estado = modelo(entrada)  # procesamos la semilla para construir el estado inicial
    ultimo_caracter = entrada[:, -1:]
    generado = texto_inicial

    for _ in range(n_caracteres):
        # TODO: pasa ultimo_caracter y estado por el modelo para obtener los
        # logits del siguiente caracter y el estado actualizado de la LSTM
        logits, estado = ...

        # TODO: aplica softmax con temperatura sobre los logits del ultimo paso
        # (logits[:, -1, :] dividido entre temperatura) para obtener una
        # distribucion de probabilidad sobre el vocabulario
        probabilidades = ...

        # TODO: muestrea un indice de caracter a partir de probabilidades
        # usando torch.multinomial con num_samples=1
        idx_siguiente = ...

        generado += idx_a_char[idx_siguiente.item()]
        ultimo_caracter = idx_siguiente

    modelo.train()
    return generado


print(generar_texto(modelo, "el hub ", 120))

<details>
<summary>💡 Vedi soluzione</summary>

```python
@torch.no_grad()
def generar_texto(modelo, texto_inicial, n_caracteres, temperatura=0.8):
    modelo.eval()

    entrada = torch.tensor([[char_a_idx[c] for c in texto_inicial]], dtype=torch.long)
    _, estado = modelo(entrada)  # elaboriamo il seme per costruire lo stato iniziale
    ultimo_caracter = entrada[:, -1:]
    generado = texto_inicial

    for _ in range(n_caracteres):
        logits, estado = modelo(ultimo_caracter, estado)
        # la temperatura ammorbidisce (>1) o affila (<1) la distribuzione prima del campionamento
        probabilidades = F.softmax(logits[:, -1, :] / temperatura, dim=-1)
        idx_siguiente = torch.multinomial(probabilidades, num_samples=1)
        generado += idx_a_char[idx_siguiente.item()]
        ultimo_caracter = idx_siguiente

    modelo.train()
    return generado


print(generar_texto(modelo, "el hub ", 120))
```
</details>

## Per concludere

Hai appena costruito, pezzo per pezzo, il meccanismo che trasforma una LSTM in un generatore di testo: lo stato nascosto che riassume ciò che è stato letto finora, e il ciclo che riutilizza quello stato per prevedere un carattere dopo l'altro senza rielaborare tutto il testo dall'inizio. È la stessa logica che sostiene sistemi molto più grandi, dall'autocompletamento del telefono fino ai modelli linguistici attuali.

Prova a cambiare il seme, la temperatura o lo stesso testo di addestramento, e osserva come cambia il risultato: è il modo più rapido per sviluppare intuizione su ciò che controlla ciascuna parte del modello.

**Salva il tuo notebook con i risultati e confrontali con il resto del corso prima della prossima sessione.**